# FMNs Analysis

In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Add project root for src/ imports
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.models_and_vars import load_env_vars, ensure_nltk_path, init_spacy_and_emo
from src.io import import_fmnt_from_txt, export_df
from src.valence import fmn_valence_stats
from src.fmtools import verify_and_report_nodes
from src.analysis import (
    compare_fmns_patterns,
    compute_auras_for_words,
    keyword_compound_index,
    keyword_correlometer_bulk,
    slope_compare_groups,
)

In [2]:
# you do not need to run these download commands every time. nltk.download()
# saves these data packages to your local file system in the nltk_data directory.
# Once they are downloaded, they are available for NLTK to use in any session.

'''nltk.download('wordnet')
nltk.download('omw-1.4') # for italian and multilingual support, if needed
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng') # for word tokenization
nltk.download('averaged_perceptron_tagger') # for POS tagging
'''


"nltk.download('wordnet')\nnltk.download('omw-1.4') # for italian and multilingual support, if needed\nnltk.download('punkt')\nnltk.download('punkt_tab')\nnltk.download('averaged_perceptron_tagger_eng') # for word tokenization\nnltk.download('averaged_perceptron_tagger') # for POS tagging\n"

# Environment and models settings

In [3]:
# Set language once; everything else adapts
language = "italian"  # or "english"
env_file_path = "../env"

# Execute setup
load_env_vars(env_file_path, language=language)
ensure_nltk_path()

nlp, emo = init_spacy_and_emo(language)

# Also preload both models for explicit use
nlp_en, _ = init_spacy_and_emo("english")
nlp_it, _ = init_spacy_and_emo("italian")

Loaded SPACY_MODEL_PATH_EN: /home/franaz/coding/spacy-models/en_core_web_lg/en_core_web_lg-3.8.0
Loaded SPACY_MODEL_PATH_IT: /home/franaz/coding/spacy-models/it_core_news_lg/it_core_news_lg-3.7.0
Loaded NLTK_DATA: /home/franaz/nltk_data



/home/franaz/coding/python/projects/data_science/networks/rassereno/.rassereno-py310/lib/python3.10/site-packages/spacy/util.py:922: UserWarning: [W095] Model 'it_core_news_lg' (3.7.0) was trained with spaCy v3.7.0 and may not be 100% compatible with the current version (3.8.7). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


# fmn settings

In [4]:
# Dynamic output paths system for pipelines
base_output_path = '../out'
exported_location = f"{base_output_path}/merged-no-filters"
formatted_data_dir = 'formatting'
# Set the type of fmn to use: 'bfmns' or 'tfmns'
# (this is used for the folder name, not for the file names)
fmn_type = 'fmn'
# Keywords that we want to check for in the textual data. If not empty,
# semantic frame plots will be created for these keywords. Otherwise,
# the full FMNs will be visualized.
keywords = []
keywords = ['matematica','ansia','stress','depressione','statistica','informatica','fisica','scienza','prestazione','esame','voto','valutazione','scuola','competizione','università','calcolo','disciplina','materia']
# Dynamically extract project name from exported location and build name based on keywords
project_name = f"{exported_location.split('/')[-1]}{'_' + '_'.join(keywords) if keywords else ''}"

# Lemmatization setting
lemmatize = False

# Types of output: aggregated and individual (entire dataset vs per participant FMNs)
# Choose any subset: {'aggregated'}, {'individual'}, or both
# How to use:
# Only aggregated: methods = {'aggregated'}
# Only individual: methods = {'individual'}
# Both: methods = {'aggregated', 'individual'}
ALLOWED_METHODS = {'aggregated', 'individual'}
methods = {'aggregated'}
methods = {m.strip().lower() for m in (methods if isinstance(methods, (list, set, tuple)) else [methods])}
methods = methods & ALLOWED_METHODS
if not methods:
    raise ValueError("No valid methods selected. Use 'aggregated' and/or 'individual'.")

# Groups to subset the data (if needed). Use labels and .startswith() condition
# from the 'participant_id' column to create groups.
group_name = 'all_fmns'  # fmn name when using groups
if group_name:
    fmn_type = f'{group_name}_{fmn_type}'


files_output_dir = f'{base_output_path}/{project_name}/analysis'
figures_output_dir = f'{base_output_path}/{project_name}/figures'

# Ensure the files and figures output directories exist
os.makedirs(files_output_dir, exist_ok=True)
os.makedirs(figures_output_dir, exist_ok=True)

### Data loading and pre-processing

In [5]:
print(f"Importing fmnts from project: {exported_location}")

fmn_names = [
    "highschool_north_high_anxiety_bfmns",
    "highschool_north_low_anxiety_bfmns",
    "highschool_south_filtered_bfmns",
    "highschool_south_unfiltered_bfmns",
    "high_school_low_anxiety_gpt_oss_bfmns",
    "highschool_high_anxiety_gpt_oss_bfmns",
    "psychology_low_anxiety_gpt_oss_bfmns",
    "psychology_high_anxiety_gpt_oss_bfmns",
    "psychology_high_anxiety_bfmns",
    "psychology_low_anxiety_bfmns",
    "physics_bfmns",
    "physics_gpt_oss_bfmns",
    "researchers_filtered_bfmns",
    "researchers_unfiltered_bfmns",
]

# Construct the full paths to the .txt files, removing _bfmns or _tfmns suffixes
if methods == {'individual'}:
    fmn_paths = [
        os.path.join(exported_location, f"individual/{name}/{name.replace('_bfmns', '').replace('_tfmns', '')}.txt") for name in fmn_names
    ]
elif methods == {'aggregated'}:
    fmn_paths = [
        os.path.join(exported_location, f"aggregated/{name}/{name.replace('_bfmns', '').replace('_tfmns', '')}.txt") for name in fmn_names
    ]

print("Importing FMNs from the following paths:")
for path in fmn_paths:
    print(f" - {path}")

fmns = {}
# Additional dictionary to hold custom valences per fmn (lexicon-based when custom not available)
valences_per_fmn = {}
for path in fmn_paths:
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Expected FMN file not found: {path}")
    print(path)

    try:
        imported = import_fmnt_from_txt(
            filepath=path, language=language, custom_valences=True
        )
        # Store custom valences for this fmn
        valences_per_fmn[os.path.basename(os.path.dirname(path))] = imported.custom_valences
    except Exception as e:
        print(f"[WARN] Could not import with custom valences: {e}")
        imported = import_fmnt_from_txt(
            filepath=path, language=language, custom_valences=False
        )
        # Store custom valences for this fmn (as lexicon-based)
        valences_per_fmn[os.path.basename(os.path.dirname(path))] = imported.custom_valences

    # Use folder name as key; store the FormamentisNetwork object
    title = os.path.basename(os.path.dirname(path))
    fmns[title] = imported.fmnt

Importing fmnts from project: ../out/merged-no-filters
Importing FMNs from the following paths:
 - ../out/merged-no-filters/aggregated/highschool_north_high_anxiety_bfmns/highschool_north_high_anxiety.txt
 - ../out/merged-no-filters/aggregated/highschool_north_low_anxiety_bfmns/highschool_north_low_anxiety.txt
 - ../out/merged-no-filters/aggregated/highschool_south_filtered_bfmns/highschool_south_filtered.txt
 - ../out/merged-no-filters/aggregated/highschool_south_unfiltered_bfmns/highschool_south_unfiltered.txt
 - ../out/merged-no-filters/aggregated/high_school_low_anxiety_gpt_oss_bfmns/high_school_low_anxiety_gpt_oss.txt
 - ../out/merged-no-filters/aggregated/highschool_high_anxiety_gpt_oss_bfmns/highschool_high_anxiety_gpt_oss.txt
 - ../out/merged-no-filters/aggregated/psychology_low_anxiety_gpt_oss_bfmns/psychology_low_anxiety_gpt_oss.txt
 - ../out/merged-no-filters/aggregated/psychology_high_anxiety_gpt_oss_bfmns/psychology_high_anxiety_gpt_oss.txt
 - ../out/merged-no-filters/aggr

FileNotFoundError: Expected FMN file not found: ../out/merged-no-filters/aggregated/highschool_north_high_anxiety_bfmns/highschool_north_high_anxiety.txt

## Run the pipeline on preprocessed data

### Aggregated pipeline

In [ ]:
# Method to execute: a single aggregated BFMN from all data
if 'aggregated' in methods:
    # Monitor initial memory usage
    print("Starting aggregated pipeline...\n")

    # Verify presence of keywords in each FMN
    verify_and_report_nodes(
        fmns=fmns,
        required=keywords,
        strict=True
    )

    monitor_memory()

    for group_name, fmn in fmns.items():
        # fmn custom valences extraction for the fmn
        custom_valences = valences_per_fmn.get(group_name, None)
        valence_stats = fmn_valence_stats(fmn, custom_valences)

        # Inspect the FMN
        inspect_fmn(
            fmn=fmn,
            participant=group_name,
            all_features={group_name: valence_stats},
            emo=emo
        )

# Deep patterns analysis

In [ ]:
results = compare_fmns_patterns(
    fmns=fmns,
    language=language,
    custom_valences=globals().get("custom_valences"),  # or None to use lexicon
    top_k=25,
    min_edge_freq=2,  # show edges appearing in at least 2 FMNs
)

In [ ]:
# Batch on your keywords for the first FMN
auras_df = compute_auras_for_words(list(fmns.values())[0], keywords, emo=emo,
                                   custom_valences=globals().get("custom_valences"))
display(auras_df.head(20))

In [ ]:
# Bulk correlometer across all FMNs and keywords
bulk = keyword_correlometer_bulk(
    fmns=fmns,
    keywords=keywords,
    emo=emo,
    per_fmn_custom_valences=valences_per_fmn,  # or None to use lexicon
    include_target=False,
    annotate=False,  # was True; disable to reduce clutter
)
display(bulk["df"].head(20))

In [ ]:
# Annotate only extremes on the correlometer (optional)
df = bulk["df"].copy()
ax = bulk["ax"]

K = 8  # number of labels to show from each criterion
top_imp = df.nlargest(K, "importance")
top_abs_val = df.assign(ext=np.abs(df["valence_score"])).nlargest(K, "ext")
to_annot = pd.concat([top_imp, top_abs_val]).drop_duplicates(subset=["group","keyword"])

for _, r in to_annot.iterrows():
    ax.annotate(
        r["keyword"],
        (r["importance"], r["valence_score"]),
        textcoords="offset points",
        xytext=(6, 5),
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.65),
        zorder=5,
    )
bulk["fig"].canvas.draw_idle()

# Export correlometer figure
corr_fig_path = os.path.join(figures_output_dir, "keyword_correlometer.png")
bulk["fig"].savefig(corr_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved correlometer figure to {corr_fig_path}")

In [ ]:
# Compound index for all keywords across all FMNs
result = keyword_compound_index(
    fmns=fmns,
    keywords=keywords,
    emo=emo,
    per_fmn_custom_valences=valences_per_fmn,
)

# Export result["wide"] for reporting
out_csv = os.path.join(base_output_path, project_name, "aggregated", fmn_type, "compound_index_wide.csv")
os.makedirs(os.path.dirname(out_csv), exist_ok=True)
result["wide"].to_csv(out_csv)
print(f"Saved: {out_csv}")

display(result["df"].head(20))
display(result["wide"].head())

In [ ]:
df = bulk["df"].copy()

# Faceted correlometer per FMN group
g = sns.relplot(
    data=df,
    x="importance", y="valence_score",
    hue="aura_label",
    col="group", col_wrap=4,
    kind="scatter",
    palette={"positive":"#3b82f6","neutral":"#9ca3af","negative":"#ef4444"},
    edgecolor="black", linewidth=0.4, alpha=0.9,
    height=3.2, aspect=1.05,
)

g.map_dataframe(_add_labels)

for ax in g.axes.flat:
    ax.axvline(0.5, color="#d1d5db", ls="--", lw=0.8)
    ax.axhline(0.0, color="#d1d5db", ls="--", lw=0.8)
    ax.set_xlim(0,1); ax.set_ylim(-1,1)
g.set_titles(col_template="{col_name}")
g.figure.tight_layout()

# Export faceted figure
facet_fig_path = os.path.join(figures_output_dir, "keyword_correlometer_faceted.png")
g.figure.savefig(facet_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved faceted correlometer figure to {facet_fig_path}")

In [ ]:
# 2) Heatmaps: keywords × groups for valence and importance
df = bulk["df"].copy()
val_pivot = df.pivot(index="keyword", columns="group", values="valence_score").sort_index()
imp_pivot = df.pivot(index="keyword", columns="group", values="importance").sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, max(5, 0.35*len(val_pivot))), dpi=130, constrained_layout=True)
sns.heatmap(val_pivot, ax=axes[0], cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.2, linecolor="#eeeeee", cbar_kws={"label":"valence_score"})
axes[0].set_title("Valence score (pos_share − neg_share)")
sns.heatmap(imp_pivot, ax=axes[1], cmap="YlGnBu", vmin=0, vmax=1,
            linewidths=0.2, linecolor="#eeeeee", cbar_kws={"label":"importance"})
axes[1].set_title("Importance (0–1)")
for ax in axes: ax.set_xlabel(""); ax.set_ylabel("")
plt.show()

# Optional: clustered heatmap to reveal blocks
g = sns.clustermap(val_pivot.fillna(0), cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                   figsize=(10,10), metric="euclidean", method="average")
g.figure.suptitle("Clustered valence patterns"); plt.show()

# Export heatmap figure
heatmap_fig_path = os.path.join(figures_output_dir, "keyword_heatmaps.png")
fig.savefig(heatmap_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved heatmap figure to {heatmap_fig_path}")

In [ ]:
# 3) Outliers: most central and most extreme valence
df = bulk["df"].copy()
top_extreme = (
    df.assign(ext=np.abs(df["valence_score"]))
      .sort_values(["importance","ext"], ascending=[False, False])
      .head(20)[["group","keyword","importance","valence_score","aura_label","n_neighbors"]]
)
display(top_extreme)

In [ ]:
# 4) Aura distribution per group (across your keywords)
df = bulk["df"].copy()
counts = df.groupby(["group","aura_label"]).size().unstack(fill_value=0).reindex(columns=["positive","neutral","negative"])
counts = counts.div(counts.sum(axis=1), axis=0)
ax = counts.plot(kind="bar", stacked=True, figsize=(10,4),
                 color={"positive":"#3b82f6","neutral":"#9ca3af","negative":"#ef4444"})
ax.set_ylabel("Share"); ax.set_title("Aura distribution across keywords by group")
plt.tight_layout(); plt.show()

# Export aura distribution figure
aura_dist_fig_path = os.path.join(figures_output_dir, "keyword_aura_distribution.png")
fig = ax.get_figure()
fig.savefig(aura_dist_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved aura distribution figure to {aura_dist_fig_path}")

In [ ]:
# 5) Stable vocabulary across FMNs (from compare_fmns_patterns results)
nodes_df = results["nodes_df"]
stable = (
    nodes_df[(nodes_df["share"]>=0.5) & (nodes_df["dominance_ratio"]>=0.8)]
    .sort_values(["share","dominance_ratio"], ascending=False)
    .head(50)
)
display(stable)

**Connection scaled** is a per-keyword, per-FMN normalized measure of how connected that node is inside that FMN.

How to interpret:

0.0 = least connected (within that FMN), 1.0 = most connected (within that FMN).
It's a relative (within-group) connectivity score, not an absolute network-wide metric.
The companion column connection_kind_per_group tells whether the raw connection is "degree" or "strength".
Notes / caveats:

node_strengths implementation adds each edge weight to both endpoints (undirected interpretation).
If you want a global scale across FMNs, set normalize_within_fmn=False and scale externally.

In [ ]:
# 6) Slopegraph: compare two groups for a metric (from result["df"])
# Usage example
g1 = "highschool_north_low_anxiety_bfmns"
g2 = "highschool_north_high_anxiety_bfmns"
g3 = "psychology_low_anxiety_bfmns"
g4 = "psychology_high_anxiety_bfmns"
g5 = "highschool_south_unfiltered_bfmns"
g6 = "physics_bfmns"
g7 = "researchers_filtered_bfmns"
fig, ax = slope_compare_groups(
    result["df"],
    g1,
    g2,
    metric="connection_scaled",
    top_k=25,
)

# Export slopegraph figure using the same names passed to the function
slope_fig_path = os.path.join(figures_output_dir, f"{g1}_vs_{g2}.png")
fig.savefig(slope_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved slopegraph figure to {slope_fig_path}")

In [ ]:
fig, ax = slope_compare_groups(
    result["df"],
    g3,
    g4,
    metric="connection_scaled",
    top_k=25,
)

# Export slopegraph figure using the same names passed to the function
slope_fig_path = os.path.join(figures_output_dir, f"{g3}_vs_{g4}.png")
fig.savefig(slope_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved slopegraph figure to {slope_fig_path}")

In [ ]:
fig, ax = slope_compare_groups(
    result["df"],
    g3,
    g5,
    metric="connection_scaled",
    top_k=25,
)

# Export slopegraph figure using the same names passed to the function
slope_fig_path = os.path.join(figures_output_dir, f"{g3}_vs_{g5}.png")
fig.savefig(slope_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved slopegraph figure to {slope_fig_path}")

In [ ]:
fig, ax = slope_compare_groups(
    result["df"],
    g5,
    g6,
    metric="connection_scaled",
    top_k=25,
)

# Export slopegraph figure using the same names passed to the function
slope_fig_path = os.path.join(figures_output_dir, f"{g5}_vs_{g6}.png")
fig.savefig(slope_fig_path, dpi=300, bbox_inches="tight", pad_inches=0.1, facecolor="white")
print(f"Saved slopegraph figure to {slope_fig_path}")